# MorphPT Dataset Subset Summary for Paper

This notebook summarizes how the MorphPT train-test subset was selected and provides tables needed for reporting:
- Original dataset size
- Excluded tissues and used tissues
- 28 used tissues with per-tissue sizes
- Class coverage and class sizes
- Optional raw label scan to justify removing unknown and tiny classes (for example Erythrocytes, OPCs)

In [ ]:
from __future__ import annotations

import json
from collections import Counter
from glob import glob
from pathlib import Path

import pandas as pd

# Core paths (edit only if your project layout changes)
PROJECT_ROOT = Path('/hpc/group/jilab/rz179/MorphPT_MOE')
SPLIT_DIR = PROJECT_ROOT / 'prepared' / 'splits_v2_seed1337_nobreast'
MANIFEST_PATH = SPLIT_DIR / 'split_manifest_seed1337.json'
MEMMAP_META_PATH = PROJECT_ROOT / 'cache_224_splits_v2_seed1337_nobreast' / 'router_shards_meta.json'
RAW_PARQUET_DIR = PROJECT_ROOT / 'prepared' / 'shards_multiview_parquet'

# Write outputs to tc459-owned location (avoid permission errors in rz179 paths)
OUTPUT_DIR = Path('/hpc/group/jilab/tc459/MorphPT/notebooks/outputs_dataset_info')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert MANIFEST_PATH.exists(), f'Missing: {MANIFEST_PATH}'
assert MEMMAP_META_PATH.exists(), f'Missing: {MEMMAP_META_PATH}'
assert RAW_PARQUET_DIR.exists(), f'Missing: {RAW_PARQUET_DIR}'

manifest = json.loads(MANIFEST_PATH.read_text())
memmap_meta = json.loads(MEMMAP_META_PATH.read_text())

print('Loaded manifest:', MANIFEST_PATH)
print('Loaded memmap meta:', MEMMAP_META_PATH)
print('Output directory:', OUTPUT_DIR)

Loaded manifest: /hpc/group/jilab/rz179/MorphPT_MOE/prepared/splits_v2_seed1337_nobreast/split_manifest_seed1337.json
Loaded memmap meta: /hpc/group/jilab/rz179/MorphPT_MOE/cache_224_splits_v2_seed1337_nobreast/router_shards_meta.json
Output directory: /hpc/group/jilab/tc459/MorphPT/notebooks/outputs_dataset_info


In [2]:
# High-level dataset sizes
slides_ok = [s for s in manifest['slides'] if s.get('status') == 'ok']
excluded_tissues = manifest.get('excluded_tissues', [])

used_total_after_filtering_before_sampling = sum(
    sum(v for v in s['patch_counts'].values()) for s in slides_ok
)
train_pool_before_router_sampling = sum(
    sum(s['patch_counts'][str(i)] for i in s['train_patches']) for s in slides_ok
)
test_total = manifest['counts']['test_total']
router_total = manifest['counts']['router_total']

summary = pd.DataFrame(
    [
        ['Raw tissues discovered', manifest['counts']['n_slides_ok'] + manifest['counts']['n_slides_excluded']],
        ['Used tissues', len(slides_ok)],
        ['Excluded tissues', len(excluded_tissues)],
        ['Cells retained after filtering (before router sampling)', used_total_after_filtering_before_sampling],
        ['Train pool before router sampling', train_pool_before_router_sampling],
        ['Router sampled train cells', router_total],
        ['Test cells', test_total],
        ['Memmap n (router)', memmap_meta['n']],
    ],
    columns=['Metric', 'Value']
)
summary

,Metric,Value
0,Raw tissues discovered,34
1,Used tissues,28
2,Excluded tissues,6
3,Cells retained after filtering (before router ...,6527726
4,Train pool before router sampling,5284657
5,Router sampled train cells,726518
6,Test cells,1243069
7,Memmap n (router),726518


In [3]:
# 28 used tissues and size of each tissue
rows = []
for s in slides_ok:
    tissue_total = sum(v for v in s['patch_counts'].values())
    train_pool = sum(s['patch_counts'][str(i)] for i in s['train_patches'])
    test_pool = sum(s['patch_counts'][str(i)] for i in s['test_patches'])
    rows.append({
        'tissue': s['tissue'],
        'total_cells_used': tissue_total,
        'train_pool_cells': train_pool,
        'test_cells': test_pool,
        'router_sampled_train_cells': s['n_router'],
    })

tissue_df = pd.DataFrame(rows).sort_values('total_cells_used', ascending=False).reset_index(drop=True)

print('Tissues used:', len(tissue_df))
display(tissue_df)

# Optional export for supplementary tables
out_csv = OUTPUT_DIR / 'dataset_info_tissues_used.csv'
tissue_df.to_csv(out_csv, index=False)
print('Saved:', out_csv)

Tissues used: 28


,tissue,total_cells_used,train_pool_cells,test_cells,router_sampled_train_cells
0,Xenium_V1_hTonsil_reactive_follicular_hyperplasia,1229583,1009874,219709,36755
1,Xenium_V1_hTonsil_follicular_lymphoid_hyperplasia,746476,588033,158443,23692
2,Xenium_human_Lymph_Node_FFPE,662008,547904,114104,50738
3,Xenium_V1_hColon_Cancer_Add_on,495994,416514,79480,45661
4,Xenium_V1_hColon_Cancer_Base,491489,416089,75400,47371
5,Xenium_Preview_Human_Lung_Cancer,447257,378353,68904,43379
6,Xenium_Prime_Ovarian_Cancer_FFPE,376908,327174,49734,48260
7,Xenium_V1_hLymphNode_nondiseased,276793,223682,53111,35044
8,Xenium_Preview_Human_Non_diseased_Lung,242981,201010,41971,27185
9,Xenium_V1_hColon_Non_diseased_Base,195497,149602,45895,31372


Saved: /hpc/group/jilab/tc459/MorphPT/notebooks/outputs_dataset_info/dataset_info_tissues_used.csv


In [4]:
# Class information (coarse and fine) from split manifest
fine_classes = manifest['fine_classes']
coarse_classes = manifest['coarse_classes']
fine_alloc_totals = manifest.get('fine_alloc_totals', {})
router_fine = manifest['distribution']['router_fine']
test_fine = manifest['distribution']['test_fine']
fine_to_coarse_path = SPLIT_DIR / 'fine_to_coarse.json'
fine_to_coarse = json.loads(fine_to_coarse_path.read_text())

print('Coarse classes (n=%d):' % len(coarse_classes), coarse_classes)
print('Fine classes used (n=%d)' % len(fine_classes))

class_rows = []
for f in fine_classes:
    class_rows.append({
        'fine_class': f,
        'coarse_class': fine_to_coarse.get(f, 'NA'),
        'allocated_router_plan': fine_alloc_totals.get(f, 0),
        'router_observed': router_fine.get(f, 0),
        'test_observed': test_fine.get(f, 0),
    })

class_df = pd.DataFrame(class_rows).sort_values(['coarse_class', 'fine_class']).reset_index(drop=True)
display(class_df)

out_csv = OUTPUT_DIR / 'dataset_info_classes_used.csv'
class_df.to_csv(out_csv, index=False)
print('Saved:', out_csv)

Coarse classes (n=6): ['Cancer', 'Lymphoid', 'Neuroglial', 'Stem_Progenitor', 'Stromal', 'Tissue_Vascular']
Fine classes used (n=21)


,fine_class,coarse_class,allocated_router_plan,router_observed,test_observed
0,Colon cancer cells,Cancer,32000,32000,28165
1,Liver cancer cells,Cancer,8037,8037,2381
2,Lung cancer cells,Cancer,32000,32000,13325
3,Ovary cancer cells,Cancer,32000,32000,26110
4,Pancreas cancer cells,Cancer,31069,31069,10762
5,Skin cancer cells,Cancer,32000,32000,21919
6,B cells,Lymphoid,67000,67000,220242
7,NK cells,Lymphoid,67000,67000,26145
8,T cells,Lymphoid,67000,67000,216329
9,Astrocytes,Neuroglial,5107,5107,2325


Saved: /hpc/group/jilab/tc459/MorphPT/notebooks/outputs_dataset_info/dataset_info_classes_used.csv


In [6]:
# Raw dataset table for each fine class (before split filtering/capping)
import pyarrow.parquet as pq

fine_set = set(fine_classes)
raw_counter = Counter()
raw_total_all_labels = 0

raw_files = sorted(glob(str(RAW_PARQUET_DIR / 'tissue=*' / 'part.parquet')))
for f in raw_files:
    # Read as a single parquet file to avoid dataset-level schema merge issues
    labels = pq.ParquetFile(f).read(columns=['label']).to_pandas()['label']
    raw_total_all_labels += len(labels)
    raw_counter.update(labels[labels.isin(fine_set)].value_counts().to_dict())

raw_total_fine_only = sum(raw_counter.values())

raw_rows = []
for fine in fine_classes:
    cnt = int(raw_counter.get(fine, 0))
    raw_rows.append({
        'fine_class': fine,
        'raw_count': cnt,
        'raw_percent_within_21_fine': (100.0 * cnt / raw_total_fine_only) if raw_total_fine_only else 0.0,
        'raw_percent_of_all_labels': (100.0 * cnt / raw_total_all_labels) if raw_total_all_labels else 0.0,
    })

raw_fine_df = pd.DataFrame(raw_rows).sort_values('raw_count', ascending=False).reset_index(drop=True)

print('Raw files scanned:', len(raw_files))
print('Raw total cells (all labels):', raw_total_all_labels)
print('Raw total cells (21 fine classes only):', raw_total_fine_only)
display(raw_fine_df)

out_csv = OUTPUT_DIR / 'dataset_info_raw_fine_classes.csv'
raw_fine_df.to_csv(out_csv, index=False)
print('Saved:', out_csv)

Raw files scanned: 34
Raw total cells (all labels): 9545565
Raw total cells (21 fine classes only): 9208928


,fine_class,raw_count,raw_percent_within_21_fine,raw_percent_of_all_labels
0,Epithelial cells,1951680,21.193346,20.445935
1,T cells,1319052,14.323622,13.818480
2,Myeloid cells,1309279,14.217496,13.716097
3,B cells,1089992,11.836253,11.418832
4,Fibroblasts,951877,10.336458,9.971929
5,Endothelial cells,631039,6.852470,6.610808
6,Stem and progenitor cells,626997,6.808578,6.568464
7,Smooth muscle cells,417510,4.533752,4.373864
8,NK cells,192502,2.090384,2.016664
9,Ovary cancer cells,190324,2.066734,1.993847


Saved: /hpc/group/jilab/tc459/MorphPT/notebooks/outputs_dataset_info/dataset_info_raw_fine_classes.csv


## Optional: Raw label scan to justify excluded tiny/unknown classes

Run the next cell if you want evidence from the original parquet shards before the final subset.
It computes label frequencies across all raw tissues and flags:
- Unknown-like labels (contains unknown, unassigned, or unlabeled)
- Tiny labels by absolute count
- Tiny labels by percentage

In [7]:
# This can take time depending on I/O.
# Requires pyarrow to read parquet label columns.

import re
import pyarrow.parquet as pq

ABSOLUTE_COUNT_THRESHOLD = 500
PERCENT_THRESHOLD = 0.05  # percent

counter = Counter()
raw_files = sorted(glob(str(RAW_PARQUET_DIR / 'tissue=*' / 'part.parquet')))

for f in raw_files:
    # Read as a single parquet file to avoid dataset-level schema merge issues
    labels = pq.ParquetFile(f).read(columns=['label']).to_pandas()['label']
    counter.update(labels)

total_raw = sum(counter.values())
freq_df = pd.DataFrame(
    [
        {'label': k, 'count': v, 'percent': (100.0 * v / total_raw if total_raw else 0.0)}
        for k, v in counter.items()
    ]
).sort_values('count', ascending=False).reset_index(drop=True)

unknown_mask = freq_df['label'].str.contains(r'unknown|unassigned|unlabeled', case=False, regex=True, na=False)
tiny_count_mask = freq_df['count'] < ABSOLUTE_COUNT_THRESHOLD
tiny_pct_mask = freq_df['percent'] < PERCENT_THRESHOLD

unknown_df = freq_df[unknown_mask].copy()
tiny_df = freq_df[tiny_count_mask | tiny_pct_mask].copy()

print('Raw total cells:', total_raw)
print('Distinct raw labels:', len(freq_df))
print('Unknown-like labels found:', len(unknown_df))
print('Tiny labels found (count < %d or percent < %.3f%%): %d' % (
    ABSOLUTE_COUNT_THRESHOLD, PERCENT_THRESHOLD, len(tiny_df)
))

display(freq_df)
display(unknown_df)
display(tiny_df.sort_values('count'))

# Explicit checks for common examples
for key in ['Erythrocytes', 'OPCs']:
    match = freq_df[freq_df['label'].str.lower() == key.lower()]
    if len(match):
        row = match.iloc[0]
        print(f"{key}: count={int(row['count'])}, percent={row['percent']:.4f}%")
    else:
        print(f"{key}: not found in raw labels")

Raw total cells: 9545565
Distinct raw labels: 28
Unknown-like labels found: 0
Tiny labels found (count < 500 or percent < 0.050%): 5


,label,count,percent
0,Epithelial cells,1951680,20.445935
1,T cells,1319052,13.818480
2,Myeloid cells,1309279,13.716097
3,B cells,1089992,11.418832
4,Fibroblasts,951877,9.971929
5,Endothelial cells,631039,6.610808
6,Stem and progenitor cells,626997,6.568464
7,Smooth muscle cells,417510,4.373864
8,Breast cancer cells,316275,3.313319
9,NK cells,192502,2.016664


,label,count,percent


,label,count,percent
27,Erythrocytes,497,0.005207
26,Kidney cancer cells,1128,0.011817
25,Brain cancer cells,2344,0.024556
24,OPCs,2488,0.026064
23,Cardiac muscle cells,3472,0.036373


Erythrocytes: count=497, percent=0.0052%
OPCs: count=2488, percent=0.0261%


In [9]:
# Expert training set size exploration (raw shards vs capped shards)
expert_dirs = sorted([p for p in SPLIT_DIR.glob('expert_*') if p.is_dir()])

expert_summary_rows = []
expert_class_rows = []
expert_tissue_rows = []

for expert_dir in expert_dirs:
    expert_name = expert_dir.name.replace('expert_', '')
    for split_name in ['shards', 'shards_capped']:
        split_dir = expert_dir / split_name
        if not split_dir.exists():
            continue

        shard_files = sorted(split_dir.glob('*.parquet'))
        if not shard_files:
            continue

        class_counter = Counter()
        tissue_counter = Counter()
        total_cells = 0

        for pf in shard_files:
            df_local = pd.read_parquet(pf, columns=['label', 'tissue'])
            n = len(df_local)
            total_cells += n
            class_counter.update(df_local['label'].value_counts().to_dict())
            tissue_counter.update(df_local['tissue'].value_counts().to_dict())

        n_classes = len(class_counter)
        n_tissues = len(tissue_counter)

        expert_summary_rows.append({
            'expert': expert_name,
            'split': split_name,
            'n_files': len(shard_files),
            'n_tissues': n_tissues,
            'n_classes': n_classes,
            'total_cells': total_cells,
        })

        for lbl, cnt in sorted(class_counter.items(), key=lambda x: (-x[1], x[0])):
            expert_class_rows.append({
                'expert': expert_name,
                'split': split_name,
                'fine_class': lbl,
                'count': int(cnt),
                'percent_within_split': 100.0 * cnt / total_cells if total_cells else 0.0,
            })

        for tissue, cnt in sorted(tissue_counter.items(), key=lambda x: (-x[1], x[0])):
            expert_tissue_rows.append({
                'expert': expert_name,
                'split': split_name,
                'tissue': tissue,
                'count': int(cnt),
                'percent_within_split': 100.0 * cnt / total_cells if total_cells else 0.0,
            })

expert_summary_df = pd.DataFrame(expert_summary_rows).sort_values(
    ['expert', 'split']
).reset_index(drop=True)
expert_class_df = pd.DataFrame(expert_class_rows).sort_values(
    ['expert', 'split', 'count'], ascending=[True, True, False]
).reset_index(drop=True)
expert_tissue_df = pd.DataFrame(expert_tissue_rows).sort_values(
    ['expert', 'split', 'count'], ascending=[True, True, False]
).reset_index(drop=True)

print('Expert splits found:', len(expert_summary_df))
display(expert_summary_df)

print('\nPer-expert class distribution (top rows):')
display(expert_class_df.head(50))

print('\nPer-expert tissue distribution (top rows):')
display(expert_tissue_df.head(50))

# Pivot view to compare shards vs shards_capped total size side-by-side
if not expert_summary_df.empty:
    pivot_total = expert_summary_df.pivot_table(
        index='expert', columns='split', values='total_cells', aggfunc='first'
    ).reset_index()
    print('\nTotal cells by expert and split:')
    display(pivot_total)

# Export tables
expert_summary_csv = OUTPUT_DIR / 'dataset_info_expert_split_summary.csv'
expert_class_csv = OUTPUT_DIR / 'dataset_info_expert_split_classes.csv'
expert_tissue_csv = OUTPUT_DIR / 'dataset_info_expert_split_tissues.csv'

expert_summary_df.to_csv(expert_summary_csv, index=False)
expert_class_df.to_csv(expert_class_csv, index=False)
expert_tissue_df.to_csv(expert_tissue_csv, index=False)

print('Saved:', expert_summary_csv)
print('Saved:', expert_class_csv)
print('Saved:', expert_tissue_csv)

Expert splits found: 5


,expert,split,n_files,n_tissues,n_classes,total_cells
0,Cancer,shards,11,11,6,241633
1,Lymphoid,shards,25,25,3,691176
2,Neuroglial,shards,3,3,4,38271
3,Tissue_Vascular,shards,28,28,6,1430241
4,Tissue_Vascular,shards_capped,28,28,6,653852



Per-expert class distribution (top rows):


,expert,split,fine_class,count,percent_within_split
0,Cancer,shards,Colon cancer cells,60000,24.831045
1,Cancer,shards,Lung cancer cells,52645,21.787173
2,Cancer,shards,Skin cancer cells,51106,21.150257
3,Cancer,shards,Pancreas cancer cells,39845,16.489883
4,Cancer,shards,Ovary cancer cells,30000,12.415523
5,Cancer,shards,Liver cancer cells,8037,3.326119
6,Lymphoid,shards,T cells,365440,52.872206
7,Lymphoid,shards,B cells,243372,35.211292
8,Lymphoid,shards,NK cells,82364,11.916502
9,Neuroglial,shards,Oligodendrocytes,17216,44.984453



Per-expert tissue distribution (top rows):


,expert,split,tissue,count,percent_within_split
0,Cancer,shards,Xenium_Prime_Ovarian_Cancer_FFPE,30000,12.415523
1,Cancer,shards,Xenium_V1_hColon_Cancer_Add_on,30000,12.415523
2,Cancer,shards,Xenium_V1_hColon_Cancer_Base,30000,12.415523
3,Cancer,shards,Xenium_V1_hLung_cancer,30000,12.415523
4,Cancer,shards,Xenium_V1_hSkin_Melanoma_Base,28937,11.975599
5,Cancer,shards,Xenium_V1_hPancreas_Cancer_Add_on,28776,11.908969
6,Cancer,shards,Xeniumranger_V1_hSkin_Melanoma_Add_on,22169,9.174657
7,Cancer,shards,Xenium_human_Lung_Cancer_FFPE,14239,5.892821
8,Cancer,shards,Xenium_human_Pancreas_FFPE,11069,4.580914
9,Cancer,shards,Xenium_Preview_Human_Lung_Cancer,8406,3.478829



Total cells by expert and split:


split,expert,shards,shards_capped
0,Cancer,241633.0,NaN
1,Lymphoid,691176.0,NaN
2,Neuroglial,38271.0,NaN
3,Tissue_Vascular,1430241.0,653852.0


Saved: /hpc/group/jilab/tc459/MorphPT/notebooks/outputs_dataset_info/dataset_info_expert_split_summary.csv
Saved: /hpc/group/jilab/tc459/MorphPT/notebooks/outputs_dataset_info/dataset_info_expert_split_classes.csv
Saved: /hpc/group/jilab/tc459/MorphPT/notebooks/outputs_dataset_info/dataset_info_expert_split_tissues.csv


In [8]:
# Paper-ready method paragraph generator
used_tissues = sorted(tissue_df['tissue'].tolist())

method_text = (
    f"We built the MorphPT train-test subset from Xenium parquet shards by first excluding pre-defined breast tissues "
    f"(n={len(excluded_tissues)} slides) and retaining {len(used_tissues)} tissues. "
    f"Cells with labels outside the curated fine-class ontology were removed, yielding "
    f"{used_total_after_filtering_before_sampling:,} cells before train sampling. "
    f"For spatial splitting, each slide was partitioned into a 5x5 grid and 5 spatially separated patches were held out as test. "
    f"This produced {test_total:,} test cells and a train pool of {train_pool_before_router_sampling:,} cells. "
    f"Router training data were then class-balanced with per-slide floor/cap constraints to {router_total:,} cells. "
    f"The final subset contained {len(coarse_classes)} coarse classes and {len(fine_classes)} fine classes."
)

print(method_text)

print('\nUsed tissues (for Supplementary):')
for t in used_tissues:
    print('-', t)

We built the MorphPT train-test subset from Xenium parquet shards by first excluding pre-defined breast tissues (n=6 slides) and retaining 28 tissues. Cells with labels outside the curated fine-class ontology were removed, yielding 6,527,726 cells before train sampling. For spatial splitting, each slide was partitioned into a 5x5 grid and 5 spatially separated patches were held out as test. This produced 1,243,069 test cells and a train pool of 5,284,657 cells. Router training data were then class-balanced with per-slide floor/cap constraints to 726,518 cells. The final subset contained 6 coarse classes and 21 fine classes.

Used tissues (for Supplementary):
- Xenium_Preview_Human_Lung_Cancer
- Xenium_Preview_Human_Non_diseased_Lung
- Xenium_Prime_Ovarian_Cancer_FFPE
- Xenium_V1_FFPE_Human_Brain_Alzheimers
- Xenium_V1_FFPE_Human_Brain_Glioblastoma
- Xenium_V1_FFPE_Human_Brain_Healthy
- Xenium_V1_hColon_Cancer_Add_on
- Xenium_V1_hColon_Cancer_Base
- Xenium_V1_hColon_Non_diseased_Add_on
